In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.signal as signal

class PTBXLDataset(Dataset):
    """
    Production PyTorch Dataset loading the real PTB-XL Clinical Cohort.
    
    Data Directory: data/ptb_xl/tensors/{train, val, test}
    Tensor Shapes:
        full_12_lead: Tensor of shape (12, 5000) representing 10s at 500 Hz
        input_3_lead: Tensor of shape (3, 5000) selecting Leads II, V1, V5
    """
    def __init__(self, data_dir="../data/ptb_xl", split="val"):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.split = split
        self.tensor_dir = self.data_dir / "tensors" / split
        
        # Fallback path if rendering from root or book subfolder
        if not self.tensor_dir.exists():
            self.tensor_dir = Path("data/ptb_xl/tensors") / split
            
        # Discover all precomputed 12-lead .pt files
        self.file_paths = sorted(list(self.tensor_dir.glob("*.pt")))
        if len(self.file_paths) == 0:
            raise FileNotFoundError(f"No PTB-XL tensor files found in {self.tensor_dir}")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load real 12-lead clinical tensor of shape (12, 5000)
        full_12_lead = torch.load(self.file_paths[idx], map_location="cpu")
        
        # Select 3 reduced input leads: Lead II (idx 1), V1 (idx 6), V5 (idx 10)
        input_3_lead = full_12_lead[[1, 6, 10], :] 
        
        return input_3_lead, full_12_lead

# Instantiate real PTB-XL Dataset validation split
dataset = PTBXLDataset(split="val")
print(f"==================================================")
print(f"Successfully loaded PTB-XL Validation Cohort!")
print(f"Total Patient Records: {len(dataset):,}")
x_sample, y_sample = dataset[0]
print(f"Input Patch Tensor Shape:  {tuple(x_sample.shape)}  (3 Leads x 5000 Samples)")
print(f"Target 12-Lead Tensor Shape: {tuple(y_sample.shape)} (12 Leads x 5000 Samples)")
print(f"==================================================")

Successfully loaded PTB-XL Validation Cohort!
Total Patient Records: 2,183
Input Patch Tensor Shape:  (3, 5000)  (3 Leads x 5000 Samples)
Target 12-Lead Tensor Shape: (12, 5000) (12 Leads x 5000 Samples)


In [3]:
#| label: real-ptbxl-12lead-grid
lead_names = ['Lead I', 'Lead II', 'Lead III', 'aVR', 'aVL', 'aVF', 
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

fig_12real = make_subplots(rows=6, cols=2, subplot_titles=lead_names, shared_xaxes=True)
t_10s = np.linspace(0, 10.0, 5000)

for idx, name in enumerate(lead_names):
    r_idx = (idx // 2) + 1
    c_idx = (idx % 2) + 1
    raw_sig = y_sample[idx].numpy()
    
    # Add raw trace (blue) and filtered trace (emerald)
    fig_12real.add_trace(go.Scatter(
        x=t_10s[:1500], y=raw_sig[:1500], mode='lines', 
        line=dict(color='#38bdf8', width=1), name=f'{name} Raw', showlegend=False
    ), row=r_idx, col=c_idx)

fig_12real.update_layout(
    title="Real PTB-XL Patient 12-Lead ECG Array (Patient #1, 3.0 Seconds)",
    template="plotly_dark",
    height=850,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_12real.show()

In [4]:
# Extract real Lead I signal from first patient record
fs = 500  # Sampling rate = 500 Hz
real_lead_I = y_sample[0].numpy() # Lead I (Index 0): shape (5000,)
t = np.linspace(0, len(real_lead_I) / fs, len(real_lead_I))

# Compute Discrete Fourier Transform
freqs = np.fft.fftfreq(len(real_lead_I), 1 / fs)
fft_vals = np.fft.fft(real_lead_I)
psd = np.abs(fft_vals) ** 2

# Keep positive frequencies up to Nyquist limit (250 Hz)
pos_mask = (freqs > 0) & (freqs < (fs / 2))
freqs_pos = freqs[pos_mask]
psd_pos = psd[pos_mask]

# Create interactive dual-panel Plotly figure
fig_fft = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Real PTB-XL Patient Waveform (Lead I)", "Frequency Power Spectrum (PSD)")
)

fig_fft.add_trace(
    go.Scatter(x=t[:1500], y=real_lead_I[:1500], mode='lines', name='Real ECG Signal', line=dict(color='#38bdf8')), 
    row=1, col=1
)
fig_fft.add_trace(
    go.Scatter(x=freqs_pos, y=10 * np.log10(psd_pos + 1e-10), mode='lines', name='PSD (dB)', line=dict(color='#f43f5e')), 
    row=1, col=2
)

# Highlight QRS frequency band (10-25 Hz)
fig_fft.add_vrect(
    x0=10, x1=25, fillcolor="rgba(16, 185, 129, 0.25)", 
    layer="below", line_width=0, annotation_text="QRS Band", row=1, col=2
)

fig_fft.update_layout(
    title="Real PTB-XL Patient Recording: Time vs. Frequency Domain Analysis",
    template="plotly_dark",
    xaxis_title="Time (seconds)",
    xaxis2_title="Frequency (Hz)",
    yaxis2_title="Power Spectrum (dB)",
    height=400, margin=dict(l=20, r=20, t=60, b=20)
)
fig_fft.show()

In [5]:
# Design 3rd order Butterworth bandpass filter (0.5 Hz - 45 Hz)
nyquist = 0.5 * fs
low_cutoff = 0.5 / nyquist
high_cutoff = 45.0 / nyquist

b, a = signal.butter(3, [low_cutoff, high_cutoff], btype='bandpass')

# Apply zero-phase filtering to real patient signal
filtered_lead_I = signal.filtfilt(b, a, real_lead_I)

# Visualize Raw vs Filtered Real Patient Signal
fig_filt = go.Figure()
fig_filt.add_trace(go.Scatter(
    x=t[:1500], y=real_lead_I[:1500], mode='lines', 
    name='Raw Real Signal (With Baseline Drift)', 
    line=dict(color='#94a3b8', width=1.5), opacity=0.5
))
fig_filt.add_trace(go.Scatter(
    x=t[:1500], y=filtered_lead_I[:1500], mode='lines', 
    name='Filtered Signal (Zero-Phase 0.5-45 Hz)', 
    line=dict(color='#10b981', width=2)
))

fig_filt.update_layout(
    title="Zero-Phase Butterworth Bandpass Filtering on Real PTB-XL Patient Signal",
    xaxis_title="Time (seconds)",
    yaxis_title="Amplitude (mV)",
    template="plotly_dark",
    height=400, margin=dict(l=20, r=20, t=40, b=20)
)
fig_filt.show()